In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time
from scipy.io import mmread
from scipy.sparse import csr_matrix

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, Isomap
import umap.umap_ as umap

from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (silhouette_score, calinski_harabasz_score, 
                             davies_bouldin_score, adjusted_rand_score,
                             adjusted_mutual_info_score)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
random_state = 42

In [3]:
from scipy.io import mmread
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
import numpy as np

# данные (строки - новости, столбцы - слова, значения на пересечении - кол-во слов в новости)
matrix_coo = mmread('./bbc/bbc.mtx') # (9635, 2225)
X_sparse = csr_matrix(matrix_coo).T # (2225, 9635), формат данных для более эффективной работы
X_dense_raw = X_sparse.toarray() 

# загрузка слов
terms = pd.read_csv('./bbc/bbc.terms', header=None)[0].tolist()

# загрузка классов
y_true = pd.read_csv('./bbc/bbc.classes', sep=r'\s+', header=None)[1].values


print(f"Размерность исходной матрицы: {X_sparse.shape}")
print(f"Количество классов: {len(np.unique(y_true))}")
print(f"Распределение новостей по классам: {np.bincount(y_true)}") 

Размерность исходной матрицы: (2225, 9635)
Количество классов: 5
Распределение новостей по классам: [510 386 417 511 401]


In [41]:
total_elements = X_dense_raw.size
non_zero_count = np.count_nonzero(X_dense_raw)
print(f"Всего значений:          {total_elements:,}")
print(f"Ненулевых ячеек (слов):  {non_zero_count:,}")

Всего значений:          21,437,875
Ненулевых ячеек (слов):  286,774


In [35]:
# кластеризация без понижения размерности

import pandas as pd
import numpy as np
import time

from scipy.io import mmread
from scipy.sparse import csr_matrix
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (silhouette_score, calinski_harabasz_score, 
                             adjusted_rand_score, adjusted_mutual_info_score)


def run_clustering_only(X, y_true, n_clusters=5):

    results = []
    
    clustering_methods = {
        'KMeans': KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10),
        'Agglomerative': AgglomerativeClustering(n_clusters=n_clusters, linkage='ward'),
        'GMM': GaussianMixture(n_components=n_clusters, random_state=random_state)
    }
    
    for name, cur_method in clustering_methods.items():
        print("-"*50)
        print(f"Метод {name}")
        start_time = time.time()

        # кластеризация
        labels = cur_method.fit_predict(X) # labels: каждой новости соответствует найденный кластер

        cur_method_time = time.time() - start_time
        
        print(f"Время: {cur_method_time:.2f} сек.")

        # метрики
        sil = silhouette_score(X, labels)
        ch = calinski_harabasz_score(X, labels)
        ari = adjusted_rand_score(y_true, labels)
        ami = adjusted_mutual_info_score(y_true, labels)
            
        results.append({
            'Clustering': name,
            'Silhouette': sil,
            'Calinski_Harabasz': ch,
            'ARI': ari,
            'AMI': ami,
            'Time_Clustering': cur_method_time
        })
            
    return pd.DataFrame(results)


df_raw = run_clustering_only(X_dense_raw, y_true, n_clusters=5)

display_cols = ['Clustering', 'Silhouette', 'Calinski_Harabasz', 'ARI', 'AMI', 'Time_Clustering']
print("-"*50)
print("\n\nРезультаты (только кластеризация):\n")
print(df_raw[display_cols].to_string(index=False))

--------------------------------------------------
Метод KMeans
Время: 11.70 сек.
--------------------------------------------------
Метод Agglomerative
Время: 13.82 сек.
--------------------------------------------------
Метод GMM
Время: 277.41 сек.
--------------------------------------------------


Результаты (только кластеризация):

   Clustering  Silhouette  Calinski_Harabasz      ARI      AMI  Time_Clustering
       KMeans    0.195795          35.098429 0.073954 0.200181        11.697582
Agglomerative    0.065633          37.040978 0.281192 0.465527        13.820462
          GMM    0.001623          31.852315 0.335402 0.496754       277.407597


In [ ]:
# StandardScaler
# 3 компоненты

import pandas as pd
import numpy as np
import time
import umap.umap_ as umap
import trimap
import pacmap
import phate

from scipy.io import mmread
from scipy.sparse import csr_matrix
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN 
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (silhouette_score, calinski_harabasz_score, 
                             adjusted_rand_score, adjusted_mutual_info_score)

random_state = 42

# стандартизация
scaler = StandardScaler()
X_scaled_raw = scaler.fit_transform(X_dense_raw)

def run_pipeline(X, y_true, n_clusters=5):

    results = []
    
    dr_models = {
        'PCA': PCA(n_components=3, random_state=random_state), 
        't-SNE': TSNE(n_components=3, perplexity=30, random_state=random_state), 
        'UMAP': umap.UMAP(n_components=3, n_neighbors=15, min_dist=0.1, random_state=random_state),
        'TriMAP': trimap.TRIMAP(n_dims=3, n_inliers=10, n_outliers=5), 
        'PaCMAP': pacmap.PaCMAP(n_components=3, n_neighbors=15, random_state=random_state)
    }
    
    clustering_methods = {
        'KMeans': KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10),
        'Agglomerative': AgglomerativeClustering(n_clusters=n_clusters, linkage='ward'),
        'GMM': GaussianMixture(n_components=n_clusters, random_state=random_state),
        'DBSCAN': DBSCAN(eps=0.5, min_samples=5) 
    }
    
    for dr_name, dr_model in dr_models.items():
        print("-"*50)
        print(f"Метод понижения размерности {dr_name}")

        start_time = time.time()
        
        # понижение размерности
        X_dr = dr_model.fit_transform(X)

        dr_time = time.time() - start_time
        print(f"Время dr: {dr_time:.2f} сек.")

        # кластеризация
        for cl_name, cl_method in clustering_methods.items():

            labels = cl_method.fit_predict(X_dr) # labels: каждой новости соответствует найденный кластер

            # если только 1 кластер найден, метрики не считаются
            if len(np.unique(labels)) > 1:
                sil = silhouette_score(X_dr, labels)
                ch = calinski_harabasz_score(X_dr, labels)
                ari = adjusted_rand_score(y_true, labels)
                ami = adjusted_mutual_info_score(y_true, labels)
            else:
                sil, ch, ari, ami = -1, -1, -1, -1
                
                
            results.append({
                'DR_Method': dr_name,
                'Clustering': cl_name,
                'Silhouette': sil,
                'Calinski_Harabasz': ch,
                'ARI': ari,
                'AMI': ami,
                'Time_DR': dr_time
            })
            
    return pd.DataFrame(results)


df_scaled_standardScaler = run_pipeline(X_scaled_raw, y_true, n_clusters=5)


display_cols = ['DR_Method', 'Clustering', 'Silhouette', 'Calinski_Harabasz', 'ARI', 'AMI', 'Time_DR']
print("-"*50)
print("\n\nРезультаты (нормировка + понижение размерности + кластеризация):\n")
print(df_scaled_standardScaler[display_cols].sort_values(by='ARI', ascending=False).to_string(index=False))

Note: `n_components != 2` have not been thoroughly tested.


--------------------------------------------------
Метод понижения размерности PCA
Время dr: 0.67 сек.
--------------------------------------------------
Метод понижения размерности t-SNE
Время dr: 198.75 сек.
--------------------------------------------------
Метод понижения размерности UMAP
Время dr: 19.87 сек.
--------------------------------------------------
Метод понижения размерности TriMAP
Время dr: 3.10 сек.
--------------------------------------------------
Метод понижения размерности PaCMAP
Время dr: 2.89 сек.
--------------------------------------------------


Результаты (нормировка + понижение размерности + кластеризация):

DR_Method    Clustering  Silhouette  Calinski_Harabasz       ARI       AMI    Time_DR
   PaCMAP        KMeans    0.609207        8040.672030  0.862269  0.828620   2.885525
   PaCMAP Agglomerative    0.598209        7802.104113  0.807736  0.806449   2.885525
     UMAP           GMM    0.435384        1633.717145  0.763144  0.773418  19.868552
     UMAP 

In [30]:
# визуализация (StandardScaler, 3 компоненты)
# PaCMAP KMeans

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import pacmap
from sklearn.cluster import KMeans

def plot_actual_3D_pacmap(X_data, y_true, class_names):
    
    dr_model = pacmap.PaCMAP(n_components=3, n_neighbors=15, random_state=42)
    X_3d = dr_model.fit_transform(X_data)
    
    kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_3d) # кластеризация
    
    # вывод текста при наведении мыши
    if class_names is None:
        class_names = [f"Class {i}" for i in np.unique(y_true)]
    true_labels_text = [class_names[i] for i in y_true]
    pred_labels_text = [f"Кластер {i}" for i in labels]
    
    # графики
    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{'type': 'scene'}, {'type': 'scene'}]],
        subplot_titles=(f"<b>Кластеры (PaCMAP + KMeans)</b>", 
                        "<b>Истинные классы</b>")
    )
    
    # левый
    fig.add_trace(
        go.Scatter3d(
            x=X_3d[:, 0], y=X_3d[:, 1], z=X_3d[:, 2],
            mode='markers',
            marker=dict(size=3, color=labels, colorscale='Turbo', opacity=0.8),
            text=pred_labels_text, hoverinfo='text', name='Кластеры', showlegend=False
        ), row=1, col=1
    )
    
    # правый
    fig.add_trace(
        go.Scatter3d(
            x=X_3d[:, 0], y=X_3d[:, 1], z=X_3d[:, 2],
            mode='markers',
            marker=dict(size=3, color=y_true, colorscale='Turbo', opacity=0.8),
            text=true_labels_text, hoverinfo='text', name='Истинные', showlegend=False
        ), row=1, col=2
    )
    
    scene_settings = dict(
        xaxis_title="PaCMAP 1", 
        yaxis_title="PaCMAP 2", 
        zaxis_title="PaCMAP 3",
        bgcolor="rgba(240, 240, 240, 0.9)", 
        aspectmode='cube'
    )
    
    fig.update_layout(
        width=1200, height=700, 
        scene1=scene_settings, scene2=scene_settings,
        margin=dict(l=0, r=0, b=0, t=50), 
        paper_bgcolor='white'
    )
    
    fig.show()

real_class_names = ["business", "entertainment", "politics", "sport", "tech"]
plot_actual_3D_pacmap(X_scaled_raw, y_true, real_class_names)

Note: `n_components != 2` have not been thoroughly tested.


In [24]:
# MinMaxScaler
# 3 компоненты

import pandas as pd
import numpy as np
import time
import umap.umap_ as umap
import trimap
import pacmap
import phate

from scipy.io import mmread
from scipy.sparse import csr_matrix
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN 
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (silhouette_score, calinski_harabasz_score, 
                             adjusted_rand_score, adjusted_mutual_info_score)

random_state = 42

# стандартизация
scaler_minmax = MinMaxScaler()
X_minmax = scaler_minmax.fit_transform(X_dense_raw)


def run_pipeline_minmax(X, y_true, n_clusters=5):

    results = []
    
    dr_methods = {
        'PCA': PCA(n_components=3, random_state=random_state), 
        't-SNE': TSNE(n_components=3, perplexity=30, random_state=random_state), 
        'UMAP': umap.UMAP(n_components=3, n_neighbors=15, min_dist=0.1, random_state=random_state),
        'TriMAP': trimap.TRIMAP(n_dims=3, n_inliers=10, n_outliers=5), 
        'PaCMAP': pacmap.PaCMAP(n_components=3, n_neighbors=15, random_state=random_state)
    }
    
    clustering_methods = {
        'KMeans': KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10),
        'Agglomerative': AgglomerativeClustering(n_clusters=n_clusters, linkage='ward'),
        'GMM': GaussianMixture(n_components=n_clusters, random_state=random_state),
        'DBSCAN': DBSCAN(eps=0.5, min_samples=5) 
    }
    
    for dr_name, dr_model in dr_methods.items():
        print("-"*50)
        print(f"Метод понижения размерности {dr_name}")

        start_time = time.time()

        # понижение размерности
        X_dr = dr_model.fit_transform(X)

        dr_time = time.time() - start_time
        print(f"Время dr: {dr_time:.2f} сек.")

        # кластеризация
        for cl_name, cl_model in clustering_methods.items():

            labels = cl_model.fit_predict(X_dr) # labels: каждой новости соответствует найденный кластер

            # если только 1 кластер найден, метрики не считаются
            if len(np.unique(labels)) > 1:
                sil = silhouette_score(X_dr, labels)
                ch = calinski_harabasz_score(X_dr, labels)
                ari = adjusted_rand_score(y_true, labels)
                ami = adjusted_mutual_info_score(y_true, labels)
            else:
                sil, ch, ari, ami = -1, -1, -1, -1
                
            results.append({
                'DR_Method': dr_name,
                'Clustering': cl_name,
                'Silhouette': sil,
                'Calinski_Harabasz': ch,
                'ARI': ari,
                'AMI': ami,
                'Time_DR': dr_time
            })
            
    return pd.DataFrame(results)


df_scaled_minimax = run_pipeline(X_minmax, y_true, n_clusters=5)


display_cols = ['DR_Method', 'Clustering', 'Silhouette', 'Calinski_Harabasz', 'ARI', 'AMI', 'Time_DR']
print("-"*50)
print("\n\nРезультаты (нормировка + понижение размерности + кластеризация):\n")
print(df_scaled_minimax[display_cols].sort_values(by='ARI', ascending=False).to_string(index=False))

Note: `n_components != 2` have not been thoroughly tested.


--------------------------------------------------
Метод понижения размерности PCA
Время dr: 0.63 сек.
--------------------------------------------------
Метод понижения размерности t-SNE
Время dr: 123.75 сек.
--------------------------------------------------
Метод понижения размерности UMAP
Время dr: 17.66 сек.
--------------------------------------------------
Метод понижения размерности TriMAP
Время dr: 2.69 сек.
--------------------------------------------------
Метод понижения размерности PaCMAP
Время dr: 1.86 сек.
--------------------------------------------------


Результаты (нормировка + понижение размерности + кластеризация):

DR_Method    Clustering  Silhouette  Calinski_Harabasz       ARI       AMI    Time_DR
   TriMAP           GMM    0.437706        1714.718820  0.706427  0.702456   2.686254
   PaCMAP        KMeans    0.510654        5963.375814  0.626213  0.675868   1.859823
     UMAP Agglomerative    0.350494        1338.337795  0.607734  0.686789  17.661421
     UMAP 

In [34]:
# визуализация (MinMaxScaler, 3 компоненты)
# TriMAP GMM

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import pacmap
from sklearn.cluster import KMeans

def plot_minmax_best_3D(X_data, y_true, class_names):

    dr_model = trimap.TRIMAP(n_dims=3, n_inliers=10, n_outliers=5)
    X_3d = dr_model.fit_transform(X_data)
    
    gmm = GaussianMixture(n_components=5, random_state=random_state)
    labels = gmm.fit_predict(X_3d) # кластеризация
    
    # вывод текста при наведении мыши
    if class_names is None:
        class_names = [f"Class {i}" for i in np.unique(y_true)]
    true_labels_text = [class_names[i] for i in y_true]
    pred_labels_text = [f"Кластер {i}" for i in labels]
    
    # графики
    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{'type': 'scene'}, {'type': 'scene'}]],
        subplot_titles=(f"<b>Кластеры (TriMAP + GMM)</b>", 
                        "<b>Истинные классы</b>")
    )
    
    # левый
    fig.add_trace(
        go.Scatter3d(
            x=X_3d[:, 0], y=X_3d[:, 1], z=X_3d[:, 2],
            mode='markers',
            marker=dict(size=3, color=labels, colorscale='Turbo', opacity=0.8),
            text=pred_labels_text, hoverinfo='text', name='Кластеры', showlegend=False
        ), row=1, col=1
    )
    
    # правый
    fig.add_trace(
        go.Scatter3d(
            x=X_3d[:, 0], y=X_3d[:, 1], z=X_3d[:, 2],
            mode='markers',
            marker=dict(size=3, color=y_true, colorscale='Turbo', opacity=0.8),
            text=true_labels_text, hoverinfo='text', name='Истинные', showlegend=False
        ), row=1, col=2
    )
    
  
    scene_settings = dict(
        xaxis_title="TriMAP 1", 
        yaxis_title="TriMAP 2", 
        zaxis_title="TriMAP 3",
        bgcolor="rgba(240, 240, 240, 0.9)", 
        aspectmode='cube'
    )
    
    fig.update_layout(
        width=1200, height=700, 
        scene1=scene_settings, scene2=scene_settings,
        margin=dict(l=0, r=0, b=0, t=50), 
        paper_bgcolor='white'
    )
    fig.show()

real_class_names = ["business", "entertainment", "politics", "sport", "tech"]
plot_minmax_best_3D(X_minmax, y_true, real_class_names)

In [ ]:
# StandardScaler
# 3 компоненты

import pandas as pd
import numpy as np
import time
import umap.umap_ as umap
import trimap
import pacmap
import phate

from scipy.io import mmread
from scipy.sparse import csr_matrix
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN 
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (silhouette_score, calinski_harabasz_score, 
                             adjusted_rand_score, adjusted_mutual_info_score)

random_state = 42

# стандартизация
scaler = StandardScaler()
X_scaled_raw = scaler.fit_transform(X_dense_raw)

def run_pipeline(X, y_true, n_clusters=5):

    results = []
    
    dr_models = {
        'PCA': PCA(n_components=3, random_state=random_state), 
        #'t-SNE': TSNE(n_components=3, perplexity=30, random_state=random_state), 
        'UMAP': umap.UMAP(n_components=3, n_neighbors=15, min_dist=0.1, random_state=random_state),
        'TriMAP': trimap.TRIMAP(n_dims=3, n_inliers=10, n_outliers=5), 
        'PaCMAP': pacmap.PaCMAP(n_components=3, n_neighbors=15, random_state=random_state)
    }
    
    clustering_methods = {
        'DBSCAN': DBSCAN(eps=3, min_samples=5) 
    }
    
    for dr_name, dr_model in dr_models.items():
        print("-"*50)
        print(f"Метод понижения размерности {dr_name}")

        start_time = time.time()
        
        # понижение размерности
        X_dr = dr_model.fit_transform(X)

        dr_time = time.time() - start_time
        print(f"Время dr: {dr_time:.2f} сек.")

        # кластеризация
        for cl_name, cl_method in clustering_methods.items():

            labels = cl_method.fit_predict(X_dr) # labels: каждой новости соответствует найденный кластер

            # если только 1 кластер найден, метрики не считаются
            if len(np.unique(labels)) > 1:
                sil = silhouette_score(X_dr, labels)
                ch = calinski_harabasz_score(X_dr, labels)
                ari = adjusted_rand_score(y_true, labels)
                ami = adjusted_mutual_info_score(y_true, labels)
            else:
                sil, ch, ari, ami = -1, -1, -1, -1
                
                
            results.append({
                'DR_Method': dr_name,
                'Clustering': cl_name,
                'Silhouette': sil,
                'Calinski_Harabasz': ch,
                'ARI': ari,
                'AMI': ami,
                'Time_DR': dr_time
            })
            
    return pd.DataFrame(results)


df_scaled_standardScaler = run_pipeline(X_scaled_raw, y_true, n_clusters=5)


display_cols = ['DR_Method', 'Clustering', 'Silhouette', 'Calinski_Harabasz', 'ARI', 'AMI', 'Time_DR']
print("-"*50)
print("\n\nРезультаты (нормировка + понижение размерности + кластеризация):\n")
print(df_scaled_standardScaler[display_cols].sort_values(by='ARI', ascending=False).to_string(index=False))

Note: `n_components != 2` have not been thoroughly tested.


--------------------------------------------------
Метод понижения размерности PCA
Время dr: 0.67 сек.
--------------------------------------------------
Метод понижения размерности UMAP
Время dr: 20.23 сек.
--------------------------------------------------
Метод понижения размерности TriMAP
Время dr: 2.87 сек.
--------------------------------------------------
Метод понижения размерности PaCMAP
Время dr: 2.13 сек.
--------------------------------------------------


Результаты (нормировка + понижение размерности + кластеризация):

DR_Method Clustering  Silhouette  Calinski_Harabasz       ARI       AMI   Time_DR
   TriMAP     DBSCAN    0.253263         842.971253  0.574651  0.639144  2.867176
   PaCMAP     DBSCAN    0.690565        6426.556041  0.451523  0.654684  2.133455
      PCA     DBSCAN    0.530476         173.491646  0.000413  0.013862  0.669340
     UMAP     DBSCAN   -1.000000          -1.000000 -1.000000 -1.000000 20.234380


In [48]:
# визуализация (StandardScaler, 3 компоненты)
# PaCMAP KMeans

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import pacmap
from sklearn.cluster import KMeans

def plot_actual_3D_pacmap(X_data, y_true, class_names):
    
    dr_model = trimap.TRIMAP(n_dims=3, n_inliers=10, n_outliers=5)
    X_3d = dr_model.fit_transform(X_data)
    
    kmeans = DBSCAN(eps=3, min_samples=5) 
    labels = kmeans.fit_predict(X_3d) # кластеризация
    
    # вывод текста при наведении мыши
    if class_names is None:
        class_names = [f"Class {i}" for i in np.unique(y_true)]
    true_labels_text = [class_names[i] for i in y_true]
    pred_labels_text = [f"Кластер {i}" for i in labels]
    
    # графики
    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{'type': 'scene'}, {'type': 'scene'}]],
        subplot_titles=(f"<b>Кластеры (PaCMAP + KMeans)</b>", 
                        "<b>Истинные классы</b>")
    )
    
    # левый
    fig.add_trace(
        go.Scatter3d(
            x=X_3d[:, 0], y=X_3d[:, 1], z=X_3d[:, 2],
            mode='markers',
            marker=dict(size=3, color=labels, colorscale='Turbo', opacity=0.8),
            text=pred_labels_text, hoverinfo='text', name='Кластеры', showlegend=False
        ), row=1, col=1
    )
    
    # правый
    fig.add_trace(
        go.Scatter3d(
            x=X_3d[:, 0], y=X_3d[:, 1], z=X_3d[:, 2],
            mode='markers',
            marker=dict(size=3, color=y_true, colorscale='Turbo', opacity=0.8),
            text=true_labels_text, hoverinfo='text', name='Истинные', showlegend=False
        ), row=1, col=2
    )
    
    scene_settings = dict(
        xaxis_title="PaCMAP 1", 
        yaxis_title="PaCMAP 2", 
        zaxis_title="PaCMAP 3",
        bgcolor="rgba(240, 240, 240, 0.9)", 
        aspectmode='cube'
    )
    
    fig.update_layout(
        width=1200, height=700, 
        scene1=scene_settings, scene2=scene_settings,
        margin=dict(l=0, r=0, b=0, t=50), 
        paper_bgcolor='white'
    )
    
    fig.show()

real_class_names = ["business", "entertainment", "politics", "sport", "tech"]
plot_actual_3D_pacmap(X_scaled_raw, y_true, real_class_names)